In [1]:
import os
import re
import pandas as pd
from ase.io import read, write
from ase import Atoms

all_i = []
orblocs = []

csv = pd.read_csv('IR_all.csv')

Molnums = csv['Molecule'].tolist()

Force = csv['Force'].tolist()
Force0 = [[force] for force in Force]
Type = csv['Predicted Type'].tolist()
Type0 = [[typ] for typ in Type]

Atom1s = csv['Site1'].tolist()
Atom2s = csv['Site2'].tolist()
Broken_bonds1_1 = csv['atom1'].tolist()
Broken_bonds2_1 = csv['atom2'].tolist()
Broken_bonds1_2 = csv['atom1_2'].tolist()
Broken_bonds2_2 = csv['atom2_2'].tolist()
Broken_bonds1_3 = csv['atom1_3'].tolist()
Broken_bonds2_3 = csv['atom2_3'].tolist()
Broken_bonds1_4 = csv['atom1_4'].tolist()
Broken_bonds2_4 = csv['atom2_4'].tolist()

In [2]:
import os
import re
from ase.io import read
from ase import Atoms
import numpy as np
import random as r
import sys

def near(atomnum, atoms, tolerance=2.000, Cfirst=True):
    atoms_not_H = []
    for i, atom in enumerate(atoms):
        if atom.symbol != 'H':
            atoms_not_H.append(i)
    mole = [atomnum]
    pre_distances = atoms.get_distances(atomnum, atoms_not_H)
    distances = np.delete(pre_distances, np.where(pre_distances >= tolerance))
    distances = np.delete(distances, np.where(distances == 0))
    distances.sort()
    length = len(distances)
    if length > 4:
        distances = distances[:4]
    for i, atom in enumerate(atoms_not_H):
        distance = atoms.get_distance(atom, atomnum)
        if distance in distances:
            mole.append(atom)
    mole_C = []
    mole_other = []
    for num in mole:
        if atoms[num].symbol == 'C':
            mole_C.append(num)
        else:
            mole_other.append(num)
    if len(distances) == 1:
        if atoms[atomnum].symbol == 'C':
            return mole_C + mole_other
        else:
            return mole_other + mole_C
    if Cfirst == True:
        return mole_C + mole_other
    else:
        return mole_other + mole_C

def randomroute(file, loc='./', atomnums=[0,1], tolerance=2.000,\
             form='XYZ', Cfirst=True, ignores=[], verbose=False):
    opath = os.getcwd()
    os.chdir(loc)
    atoms = read(file+'.xyz')
    atomnum1 = atomnums[0]
    atomnum2 = atomnums[1]
    atoms_not_H = []
    atoms_H = []
    for i, atom in enumerate(atoms):
        if atom.symbol != 'H':
            atoms_not_H.append(i)
        else:
            atoms_H.append(i)
    mole = near(atomnum1, atoms, tolerance=tolerance, Cfirst=Cfirst)
    i = 1
    if verbose == True:
        print('Near:', mole)
    real_mole = [*mole]
    real_mole.remove(atomnum1)
    for num in ignores:
        try:
            real_mole.remove(num)
        except:
            pass
    if verbose == True:
        print('Real Near:', real_mole)
    ignores.append(atomnum1)
    os.chdir(opath)
    if atomnum2 in mole:
        return [atomnum1, atomnum2]
    elif len(real_mole) == 1:
        return [atomnum1, *randomroute(file, loc=loc, atomnums=[real_mole[0], atomnum2], tolerance=tolerance,\
                form=form, Cfirst=Cfirst, ignores=ignores, verbose=verbose)]
    elif len(real_mole) >= 2:
        dice = r.randint(0, len(real_mole)-1)
        if verbose == True:
            print('Dice:', dice)
        return [atomnum1, *randomroute(file, loc=loc, atomnums=[real_mole[dice], atomnum2], tolerance=tolerance,\
                form=form, Cfirst=Cfirst, ignores=ignores, verbose=verbose)]
    elif len(real_mole) == 0:
        return [-99999]

def routelength(file, a, loc='./'):
    opath = os.getcwd()
    os.chdir(loc)
    atoms = read(file+'.xyz')
    distance = 0
    for i in range(len(a)-1):
        x1 = a[0]
        x2 = a[1]
        distance = distance + atoms.get_distances(x1, x2)[0]
    os.chdir(opath)
    return distance

In [ ]:
from joblib import Parallel, delayed

loc = './Initial Structure/'

w = open('ir_route2.csv','w')
w.write("Molecule,MinRoute,MinLength,MaxRoute,MaxLength,GapRoute,GapLength\n")

minroute_lengths = []
maxroute_lengths = []
minroutes = []
maxroutes = []

key = 0

for i,num in enumerate(Molnums):
    minroute = []
    minroute_list_length = 99999
    minroute_length = 99999
    maxroute = []
    maxroute_list_length = -99999
    maxroute_length = -99999
    
    if num == '30b':
        key = 1
    if key == 0:
        continue
    
    for j,num0 in enumerate(Molnums):
        if num0 == num:
            break
    
    print('Result of {0}:'.format(num))
    
    atom1 = Atom1s[j]
    atom2 = Atom2s[j]
    print('Force Sites:',atom1, atom2)
    
    l = 0
    #while l < 4 * len(atoms1.get_positions()):
    while l < 200:
        a = randomroute(num+'_0.0', loc=loc, atomnums=[atom1, atom2], tolerance=2.000,\
                        form='XYZ', Cfirst=True, ignores=[], verbose=False)
        if -99999 not in a:
            #print('Result:', a)
            l_a = routelength(num+'_0.0', a, loc=loc)
            if l_a < minroute_length:
                minroute = a
                minroute_list_length = len(a)
                minroute_length = l_a
            if l_a > maxroute_length:
                maxroute = a
                maxroute_list_length = len(a)
                maxroute_length = l_a
            #print(len(a), 'Length:', l_a)
            l = l + 1
    
    minroutes.append(minroute)
    maxroutes.append(maxroute)
    minroute_lengths.append(minroute_length)
    maxroute_lengths.append(maxroute_length)
    
    w.write("{0},{1},{2},{3},{4},{5},{6}\n".format(num, len(minroute), minroute_length, len(maxroute), maxroute_length,\
                                                 len(maxroute)-len(minroute), maxroute_length-minroute_length))
    
    print('Min Bonds:', minroute, ',', len(minroute), 'in all' )
    print('Min Length:', minroute_length)
    print('Max Bonds:', maxroute, ',', len(maxroute), 'in all' )
    print('Max Length:', maxroute_length)
    #print("{0},{1},{2}\n".format(num, atom1, atom2))
    print("{0},{1},{2},{3},{4},{5},{6}\n".format(num, len(minroute), minroute_length, len(maxroute), maxroute_length,\
                                                 len(maxroute)-len(minroute), maxroute_length-minroute_length))
    #break
w.close()

Result of 30b:
Force Sites: 9 42


In [ ]:
w.close()

# Extra descriptors:

### CDFT test: Electrophilicity index & Nucleophilicity index

In [ ]:
command = 'printf "22\\n2\\n/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N.wfn\\n'.format(789)\
+ '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N-1.wfn\\n'.format(789)\
+ '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N+1.wfn\\n'.format(789)\
+ '0\\nq\\n" '\
+'| ./Initial\\ Structure/multiwfn-mac-build/multiwfn '\
+ '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/xyz/{0}_0.0.xyz'.format(789)

print(command)

In [ ]:
os.system(command)

In [ ]:
os.system('mv CDFT.txt ./Extra/ir/CDFT/789_CDFT.txt')

In [1]:
import os
import re
import pandas as pd
from ase.io import read, write
from ase import Atoms

In [ ]:
csv0 = pd.read_csv('route.csv')
Molnum0 = csv0['Molecule'].tolist()

mols = [Molnum0[142]]
print('Last Molecule:', mols[-1])

csv = pd.read_csv('candidate_bonds_part.csv')
print(csv)
print(Molnum0[142])

In [ ]:
Molnum0[700:][0]

In [ ]:
# Nucleophilicity index (N_Nu): E_HOMO(Nu) − E_HOMO(TCE)
# Initial E_HOMO(TCE) = -0.335198 Hartree in B3LYP/6-31G*
# E_HOMO(TCE) = -0.337331 Hartree in current computational ways

csv0 = pd.read_csv('route.csv')
Molnum0 = csv0['Molecule'].tolist()

mols = Molnum0[700:]
print('Last Molecule:', mols[-1])

csv = pd.read_csv('candidate_bonds_part.csv')
Molnum = csv['Molecule'].tolist()
a1s = csv['Atom 1'].tolist()
a2s = csv['Atom 2'].tolist()

w = open('reactivity_part.csv', 'w')
w.write('Molecule,Atom1,Elec_reac1,Nuc_reac1,Atom2,Elec_reac2,Nuc_reac2\n')

def Nuc_iRectifier(nuc_i, HOMO):
    delta_HOMO = -0.337331 - (-0.335198)
    
    real_HOMO = HOMO + delta_HOMO
    real_nuc_i = nuc_i * real_HOMO / HOMO
    return real_nuc_i

j = 0
for mol in mols:
    print('\nMolecule {0}'.format(mol))
        
    command = 'printf "22\\n2\\n/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N.wfn\\n'.format(mol)\
    + '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N+1.wfn\\n'.format(mol)\
    + '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/wfn/{0}_0.0_N-1.wfn\\n'.format(mol)\
    + '0\\nq\\n" '\
    +'| ./Initial\\ Structure/multiwfn-mac-build/multiwfn '\
    + '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/xyz/{0}_0.0.xyz'.format(mol)
    
    os.system(command)
    os.system('mv CDFT.txt ./Extra/ir/CDFT/{0}_CDFT.txt'.format(mol))
    
    f = open('./Extra/ir/CDFT/{0}_CDFT.txt'.format(mol), 'r')
    sig = 0
    ele_is = []
    nuc_is = []
    for line in f:
        if sig == 1:
            a2 = re.findall(r'\-?[0-9]+\.[0-9]+', line)
            if len(a2) == 0:
                sig = 0
                continue
            ele_is.append(eval(a2[0]))
            nuc_is.append(eval(a2[1]))
        a = re.search('Electrophilicity          Nucleophilicity', line)
        if a:
            sig = 1
        
        b = re.search(r'E_HOMO\(N\)', line)
        if b:
            b2 = re.findall(r'\-?[0-9]+\.[0-9]+', line)
            HOMO = eval(b2[0]) # Hartree
            #HOMO = eval(b2[1]) # eV
    f.close()
    print(ele_is)
    print(HOMO)
    
    delta_HOMO = -0.337331 - (-0.335198)
    real_HOMO = HOMO + delta_HOMO
    
    if type(mol) != str:
        mol = str(mol)
    
    #while Molnum[j] == mol:
    print(Molnum[j], mol)
    while str(Molnum[j]) == mol: # simply currently
        a1 = int(a1s[j])
        ele_i1 = ele_is[a1]
        nuc_i1 = nuc_is[a1]
        real_nuc_i1 = Nuc_iRectifier(nuc_i1, HOMO) * real_HOMO / HOMO
        
        a2 = int(a2s[j])
        ele_i2 = ele_is[a2]
        nuc_i2 = nuc_is[a2]
        real_nuc_i2 = Nuc_iRectifier(nuc_i2, HOMO) * real_HOMO / HOMO
        
        print('Atom {0}: Electrophilicity index: {1}, Nucleophilicity index: {2:.5f}'.format(a1, ele_i1, real_nuc_i1))
        print('Atom {0}: Electrophilicity index: {1}, Nucleophilicity index: {2:.5f}'.format(a2, ele_i2, real_nuc_i2))
        w.write('{0},{1},{2},{3:.5f},{4},{5},{6:.5f}\n'.format(mol, a1, ele_i1, real_nuc_i1, a2, ele_i2, real_nuc_i2))
        j = j + 1
    
w.close()

In [ ]:
csv = pd.read_csv('Force_candidate.csv')
Molnum = csv['Molecule'].tolist()
a1s = csv['Atom 1'].tolist()
a2s = csv['Atom 2'].tolist()

w = open('reactivity_part.csv', 'w')
w.write('Molecule,Atom1,Elec_reac1,Nuc_reac1,Atom2,Elec_reac2,Nuc_reac2\n')

def Nuc_iRectifier(nuc_i, HOMO):
    delta_HOMO = -0.337331 - (-0.335198)
    
    real_HOMO = HOMO + delta_HOMO
    real_nuc_i = nuc_i * real_HOMO / HOMO
    return real_nuc_i

for i,mol in enumerate(Molnum):
    print('\nMolecule {0}'.format(mol))
    
    f = open('./Extra/ir/CDFT/{0}_CDFT.txt'.format(mol), 'r')
    sig = 0
    ele_is = []
    nuc_is = []
    for line in f:
        if sig == 1:
            a2 = re.findall(r'\-?[0-9]+\.[0-9]+', line)
            if len(a2) == 0:
                sig = 0
                continue
            ele_is.append(eval(a2[0]))
            nuc_is.append(eval(a2[1]))
        a = re.search('Electrophilicity          Nucleophilicity', line)
        if a:
            sig = 1
        
        b = re.search(r'E_HOMO\(N\)', line)
        if b:
            b2 = re.findall(r'\-?[0-9]+\.[0-9]+', line)
            HOMO = eval(b2[0]) # Hartree
            #HOMO = eval(b2[1]) # eV
    f.close()
    #print(ele_is)
    print('HOMO:', HOMO)
    
    delta_HOMO = -0.337331 - (-0.335198)
    real_HOMO = HOMO + delta_HOMO
    
    a1 = int(a1s[i])
    ele_i1 = ele_is[a1]
    nuc_i1 = nuc_is[a1]
    real_nuc_i1 = Nuc_iRectifier(nuc_i1, HOMO) * real_HOMO / HOMO

    a2 = int(a2s[i])
    ele_i2 = ele_is[a2]
    nuc_i2 = nuc_is[a2]
    real_nuc_i2 = Nuc_iRectifier(nuc_i2, HOMO) * real_HOMO / HOMO

    print('Atom {0}: Electrophilicity index: {1}, Nucleophilicity index: {2:.5f}'.format(a1, ele_i1, real_nuc_i1))
    print('Atom {0}: Electrophilicity index: {1}, Nucleophilicity index: {2:.5f}'.format(a2, ele_i2, real_nuc_i2))
    w.write('{0},{1},{2},{3:.5f},{4},{5},{6:.5f}\n'.format(mol, a1, ele_i1, real_nuc_i1, a2, ele_i2, real_nuc_i2))

w.close()

In [ ]:
type(Molnum[j])
#119,3.7,7,7,15,21.22413187,15,21.22413187,46,53,0,1,0,11,,,,,-88.88872774,-88.88029613,-88.87608681,0.117014751,2.051377953,1.791077261,1.975442633,2.016230321,1.441617659,1.949428789,1.997587928

In [ ]:
import re
import os

# Nucleophilicity index (N_Nu): E_HOMO(Nu) − E_HOMO(TCE)
# Initial E_HOMO(TCE) = -0.335198 Hartree in B3LYP/6-31G*
# E_HOMO(TCE) = -0.337331 Hartree in current computational ways

f = open('./Extra/ir/CDFT/{0}_CDFT.txt'.format(i), 'r')
sig = 0
ele_is = []
nuc_is = []
for line in f:
    if sig == 1:
        a2 = re.findall(r'[0-9]+\.[0-9]+', line)
        if len(a2) == 0:
            sig = 0
            continue
        ele_is.append(eval(a2[0]))
        nuc_is.append(eval(a2[1]))
    a = re.search('Electrophilicity          Nucleophilicity', line)
    if a:
        sig = 1
    
    b = re.search('E_HOMO(N)', line)
    if b:
        b2 = re.findall(r'[0-9]+\.[0-9]+', line)
        HOMO = eval(b2[0]) # Hartree
        #HOMO = eval(b2[1]) # eV

real_nuc_is = [Nuc_iRectifier(i, HOMO) for i in nuc_is]
print(len(ele_is))
print(ele_is)
print(nuc_is)
print(real_nuc_is)

In [ ]:
delta_HOMO = -0.337331 - (-0.335198)

ele_i = 0.02576
nuc_i = 0.45535

HOMO = -0.278324

real_HOMO = HOMO + delta_HOMO

real_nuc_i = nuc_i * real_HOMO / HOMO

print('Electrophilicity index:', ele_i)
print('Nucleophilicity index:', real_nuc_i)

In [ ]:
! B3LYP/G 6-31G* autoaux aim
%pal nprocs   4 end
%maxcore  1000
* xyz  -1  2
C     -5.92204264    1.91237176   -1.37706181
O     -7.11089917    1.66657985   -0.78080892
C     -4.81851384    1.05853514   -0.79941207
C     -5.01761200   -0.47627719   -0.64418837
C     -4.56186458    1.12892029    0.74467341
H     -3.90100417    1.29320384   -1.35431158


In [5]:
import os
import warnings

def XYZCDFT(file, inpname='', fileloc='./', saveloc='./', method='B3LYP',\
            basis_set='def2-SVP', maxcore=-1, corenum=1, term=['N',0,1]):
    f = open(fileloc+file+'.xyz', 'r')
    if inpname == '':
        try:
            w = open(saveloc+file+'_{0}.inp'.format(term[0]), 'x')
        except:
            w = open(saveloc+file+'_{0}.inp'.format(term[0]), 'w')
    else:
        try:
            w = open(saveloc+inpname+'_{0}.inp'.format(term[0]), 'x')
        except:
            w = open(saveloc+inpname+'_{0}.inp'.format(term[0]), 'w')
    w.write('#Powered by MCPoly\n\n')
    w.write('! ')

    w.write('{0}/G {1} autoaux aim\n'.format(method, basis_set))

    if maxcore != -1:
        w.write('%maxcore {0}\n'.format(maxcore))
        if maxcore <= 2048:
            warnings.warn('Your max core space is only {0:.3f} MB.'.format(maxcore)
                + 'You might delete this key words to get the same calculation speed,'
                + 'or try bigger one.')

    if corenum != 1:
        w.write('%PAL NPROCS {0} END\n'.format(corenum))
    
    w.write('\n*xyz {0} {1}\n'.format(term[1], term[2]))
    i = 0
    for line in f:
        i = i + 1
        if i >= 3:
            w.write(line)
    w.write('*\n')
    f.close()
    w.close()

def CDFTinp(file, inpname='', fileloc='./', saveloc='./', method='B3LYP',\
            basis_set='def2-SVP', maxcore=-1, corenum=1):
    XYZCDFT(file, inpname=inpname, fileloc=fileloc, saveloc=saveloc, method=method,\
            basis_set=basis_set, maxcore=maxcore, corenum=corenum,\
            term=['N',0,1])
    XYZCDFT(file, inpname=inpname, fileloc=fileloc, saveloc=saveloc, method=method,\
            basis_set=basis_set, maxcore=maxcore, corenum=corenum,\
            term=['N+1',1,2])
    XYZCDFT(file, inpname=inpname, fileloc=fileloc, saveloc=saveloc, method=method,\
            basis_set=basis_set, maxcore=maxcore, corenum=corenum,\
            term=['N-1',-1,2])

In [ ]:
CDFTinp('2_0.0', fileloc='./Extra/', saveloc='./Extra/', method='B3LYP',\
        basis_set='def2-SVP', maxcore=4096, corenum=20)

In [6]:
import pandas as pd
import os
import re

def allbrokenbonds(mol, file='Force_candidate_transfer.csv',output='Atom'):
    csv = pd.read_csv(file)

    Molnums = csv['Molecule'].tolist()

    IsBroken = csv['IsBroken'].tolist()
    try:
        MayerBond = csv['Mayer'].tolist()
    except:
        pass
    try:
        Lengths = csv['Bond Length'].tolist()
    except:
        pass

    cand_bonds1 = csv['Atom 1'].tolist()
    cand_bonds2 = csv['Atom 2'].tolist()

    outputs = []
    for i, mol0 in enumerate(Molnums):
        if mol0 == mol:
            if IsBroken[i] == 1:
                if output == 'Atom':
                    outputs.append([cand_bonds1[i], cand_bonds2[i]])
                elif output == 'Mayer':
                    outputs.append([cand_bonds1[i], cand_bonds2[i], MayerBond[i]])
                elif output == 'Length':
                    outputs.append([cand_bonds1[i], cand_bonds2[i], Lengths[i]])
                    
    return outputs

### Mayer: Bond Type (single? double? triple? deloc?)

In [22]:
mol = '4083'

command = 'printf "9\\n1\\n'.format(mol)\
+ 'y\\n'\
+ '0\\nq\\n" '\
+'| ./Initial\\ Structure/multiwfn-mac-build/multiwfn '\
+ '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/molden/{0}_0.0.molden.input > mayer.out'.format(mol)

os.system(command)
os.system('mv mayer.out ./Extra/ir/bndmat/{0}_mayer.out'.format(mol))

Note: The following floating-point exceptions are signalling: IEEE_INVALID_FLAG IEEE_UNDERFLOW_FLAG


0

In [17]:
import pandas as pd

#csv = pd.read_csv('Force_candidate.csv')
#csv = pd.read_csv('candidate_bonds_no_conformer.csv')
csv = pd.read_csv('Force_candidate_transfer3.csv')
Molnum = csv['Molecule'].tolist()
a1s = csv['Atom 1'].tolist()
a2s = csv['Atom 2'].tolist()

w = open('Mayer_part2.csv', 'w')
w.write('Molecule,MayerBond\n')

for j, mol in enumerate(Molnum):
    #if mol == '4091':
    #    continue
    command = 'printf "9\\n1\\n'.format(mol)\
    + 'y\\n'\
    + '0\\nq\\n" '\
    +'| ./Initial\\ Structure/multiwfn-mac-build/multiwfn '\
    + '/Users/cxs454/desktop/decompoly/da/ir_b3lyp_svp/extra/ir/molden/{0}_0.0.molden.input > mayer.out'.format(mol)
    
    #os.system(command)
    #os.system('mv mayer.out ./Extra/ir/bndmat/{0}_mayer.out'.format(mol))
    
    f = open('./Extra/ir/bndmat/{0}_mayer.out'.format(mol), 'r')
    sig = 0
    atomnums = []
    mayers = []
    for line in f:
        if sig == 1:
            #print(line)
            atomnum = re.findall(r'[0-9]+', line)
            mayer = re.search(r'[0-9]+\.[0-9]+', line)
            if mayer:
                atomnums.append({int(atomnum[1]), int(atomnum[2])})
                mayers.append(eval(mayer.group(0)))
            else:
                break
        a = re.search('Bond orders with absolute value', line)
        if a:
            sig = 1
    f.close()
    
    if type(mol) != str:
        mol = str(mol)
    
    print('\nMolecule {0}'.format(mol))
    
    real_bondnum = {a1s[j]+1, a2s[j]+1}

    for i,atomnum in enumerate(atomnums):
        if real_bondnum == atomnum:
            w.write('{0},{1}\n'.format(mol,mayers[i]))
            print('Bond {0}-{1}:  {2:.5f}'.format(int(a1s[j]), int(a2s[j]), mayers[i]))
    j = j + 1
    
w.close()


Molecule 4091
Bond 0-58:  0.82518

Molecule 4091
Bond 4-0:  0.89788

Molecule 4091
Bond 58-62:  0.89809

Molecule 4091
Bond 55-51:  1.06205

Molecule 4091
Bond 109-113:  1.06192


In [16]:
#csv = pd.read_csv('all_advanced.csv')
csv = pd.read_csv('all_transfer.csv')
Molnums = csv['Molecule'].tolist()

loc = './Extra/'

w = open('standardised_bonds.csv','w')
w.write("Molecule,Mayer Bonds1,Mayer Bonds2,Mayer Bonds3,Mayer Bonds4\n")

for turn,num in enumerate(Molnums):
    bondbox = allbrokenbonds(num, output='Mayer')
    atoms = read(loc+f'{num}_0.0.xyz')
    w.write('{0},'.format(num))
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    ele1 = atoms.get_chemical_symbols()[bondnum[0]]
    ele2 = atoms.get_chemical_symbols()[bondnum[1]]
    bondele = [ele1, ele2]
    print('Molecule {0}:'.format(num))
    print('Reaction Atoms 1:', bondele, bondnum[:2])
    print('Mayer:', bondnum[-1])
    w.write('{0},'.format(bondnum[-1]))
    try:
        bondnum = bondbox[1]
        ele1 = atoms.get_chemical_symbols()[bondnum[0]]
        ele2 = atoms.get_chemical_symbols()[bondnum[1]]
        bondele = [ele1, ele2]
        print('Reaction Atoms 2:', bondele, bondnum[:2])
        print('Mayer:', bondnum[-1])
        w.write('{0},'.format(bondnum[-1]))
        try:
            bondnum = bondbox[2]
            ele1 = atoms.get_chemical_symbols()[bondnum[0]]
            ele2 = atoms.get_chemical_symbols()[bondnum[1]]
            bondele = [ele1, ele2]
            print('Reaction Atoms 3:', bondele, bondnum[:2])
            print('Mayer:', bondnum[-1])
            w.write('{0},'.format(bondnum[-1]))
            try:
                bondnum = bondbox[3]
                ele1 = atoms.get_chemical_symbols()[bondnum[0]]
                ele2 = atoms.get_chemical_symbols()[bondnum[1]]
                bondele = [ele1, ele2]
                print('Reaction Atoms 4:', bondele, bondnum[:2])
                print('Mayer:', bondnum[-1])
                w.write('{0},'.format(bondnum[-1]))
            except:
                pass
        except:
            pass
    except:
        pass
    print()
    w.write('\n')

w.close()

Molecule 4005:
Reaction Atoms 1: ['C', 'C'] [6, 2]
Mayer: 0.967469

Molecule 4008:
Reaction Atoms 1: ['C', 'C'] [2, 0]
Mayer: 0.92614059

Molecule 4009:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.82710298

Molecule 4009-2:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.83050718

Molecule 4010:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.81247495

Molecule 4010-2:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.81199942

Molecule 4012:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.78045164

Molecule 4012-2:
Reaction Atoms 1: ['C', 'C'] [7, 8]
Mayer: 0.82753656

Molecule 4013:
Reaction Atoms 1: ['C', 'C'] [4, 1]
Mayer: 0.96769901

Molecule 4014:
Reaction Atoms 1: ['C', 'C'] [1, 4]
Mayer: 0.95058875

Molecule 4015:
Reaction Atoms 1: ['C', 'C'] [4, 1]
Mayer: 0.96326053

Molecule 4016:
Reaction Atoms 1: ['O', 'C'] [18, 17]
Mayer: 0.92988269

Molecule 4017:
Reaction Atoms 1: ['O', 'C'] [18, 17]
Mayer: 0.93845329

Molecule 4018:
Reaction Atoms 1: ['O', 'C'] [20, 19]
Mayer: 0.92450692

Molecule 4

In [ ]:
### Machine Learning Model

import os
import re
from ase.io import read
from ase import Atoms
import numpy as np
import pandas as pd
import random as r
import sys

csv = pd.read_csv('all.csv')

Molnums = csv['Molecule'].tolist()

Broken_bonds1_1 = csv['atom1'].tolist()
Broken_bonds2_1 = csv['atom2'].tolist()
Broken_bonds1_2 = csv['atom1_2'].tolist()
Broken_bonds2_2 = csv['atom2_2'].tolist()
Broken_bonds1_3 = csv['atom1_3'].tolist()
Broken_bonds2_3 = csv['atom2_3'].tolist()
Broken_bonds1_4 = csv['atom1_4'].tolist()
Broken_bonds2_4 = csv['atom2_4'].tolist()

In [46]:
def standard_bonds_lengths(d, atoms_ele):
    CC = 1.528
    CO = 1.407
    CN = 1.457
    OO = 1.442
    NN = 1.473
    NO = 1.431
    SS = 2.100
    SO = 1.690
    SC = 1.830
    if atoms_ele == {'C', 'C'}:
        return d / CC
    elif atoms_ele == {'C', 'O'}:
        return d / CO
    elif atoms_ele == {'C', 'N'}:
        return d / CN
    elif atoms_ele == {'O', 'O'}:
        return d / OO
    elif atoms_ele == {'N', 'N'}:
        return d / NN
    elif atoms_ele == {'N', 'O'}:
        return d / NO
    elif atoms_ele == {'S', 'S'}:
        return d / SS
    elif atoms_ele == {'S', 'O'}:
        return d / SO
    elif atoms_ele == {'S', 'C'}:
        return d / SC
    else:
        return None

loc = './Extra/'

w = open('standardised_bonds.csv','w')
w.write("Molecule,Standard Bonds1,Standard Bonds2,Standard Bonds3,Standard Bonds4\n")
for turn,num in enumerate(Molnums):
    bondbox = allbrokenbonds(num)
    atoms = read(loc+f'{num}_0.0.xyz')
    w.write('{0},'.format(num))
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    ele1 = atoms.get_chemical_symbols()[bondnum[0]]
    ele2 = atoms.get_chemical_symbols()[bondnum[1]]
    bondele = [ele1, ele2]
    print('Reaction Atoms 1:', bondele, bondnum)
    d = atoms.get_distance(*bondnum)
    standard_d = standard_bonds_lengths(d, {*bondele})
    print('Length:', standard_d)
    w.write('{0},'.format(standard_d))
    try:
        bondnum = bondbox[1]
        ele1 = atoms.get_chemical_symbols()[bondnum[0]]
        ele2 = atoms.get_chemical_symbols()[bondnum[1]]
        bondele = [ele1, ele2]
        print('Reaction Atoms 2:', bondele, bondnum)
        d = atoms.get_distance(*bondnum)
        standard_d = standard_bonds_lengths(d, {*bondele})
        print('Length:', standard_d)
        w.write('{0},'.format(standard_d))
        try:
            bondnum = bondbox[2]
            ele1 = atoms.get_chemical_symbols()[bondnum[0]]
            ele2 = atoms.get_chemical_symbols()[bondnum[1]]
            bondele = [ele1, ele2]
            print('Reaction Atoms 3:', bondele, bondnum)
            d = atoms.get_distance(*bondnum)
            standard_d = standard_bonds_lengths(d, {*bondele})
            print('Length:', standard_d)
            w.write('{0},'.format(standard_d))
            try:
                bondnum = bondbox[3]
                ele1 = atoms.get_chemical_symbols()[bondnum[0]]
                ele2 = atoms.get_chemical_symbols()[bondnum[1]]
                bondele = [ele1, ele2]
                print('Reaction Atoms 4:', bondele, bondnum)
                d = atoms.get_distance(*bondnum)
                standard_d = standard_bonds_lengths(d, {*bondele})
                print('Length:', standard_d)
                w.write('{0},'.format(standard_d))
            except:
                pass
        except:
            pass
    except:
        pass
    w.write('\n')

w.close()

Reaction Atoms 1: ['C', 'C'] [6, 2]
Length: 1.041837957870983
Reaction Atoms 1: ['C', 'C'] [2, 0]
Length: 1.007550411902488
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 0.99229947639682
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 0.9921665566844946
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 1.0014418216949292
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 1.00528374091123
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 1.0014602714251026
Reaction Atoms 1: ['C', 'C'] [7, 8]
Length: 0.9892157825323039
Reaction Atoms 1: ['C', 'C'] [4, 1]
Length: 1.000847374295553
Reaction Atoms 1: ['C', 'C'] [1, 4]
Length: 0.9845776075654258
Reaction Atoms 1: ['C', 'C'] [4, 1]
Length: 0.99111695551127
Reaction Atoms 1: ['O', 'C'] [18, 17]
Length: 1.0430382253224177
Reaction Atoms 1: ['O', 'C'] [18, 17]
Length: 1.0401846536589436
Reaction Atoms 1: ['O', 'C'] [20, 19]
Length: 1.0428396227803969
Reaction Atoms 1: ['C', 'O'] [17, 18]
Length: 1.017766509551003
Reaction Atoms 1: ['C', 'O'] [17, 18]
Length: 1.050528021

### cos(theta)

In [12]:
import os
import re
import pandas as pd

#f = open('Candidates.csv', 'r')
f = open('Candidates_transfer.csv', 'r')
mols = []
lists = []

for line in f:
    if 'Molecule' in line:
        continue
    a = re.split(',', line)
    mols.append(a[0])
    a = re.search(r'\[.*?\]', line)
    if a:
        lists.append(eval(a.group(0)))
    else:
        lists.append([])

f.close()

In [13]:
def cos_angle(v1, v2):
    v1_length = (v1[0]**2 + v1[1]**2 + v1[2]**2) ** 0.5
    v2_length = (v2[0]**2 + v2[1]**2 + v2[2]**2) ** 0.5
    dot_prod = v1[0]*v2[0] + v1[1]*v2[1] + v1[2]*v2[2]
    cos = dot_prod / (v1_length * v2_length)
    return abs(cos)

def part_angle_calc(bond, route, atoms):
    for i in range(len(route)-1):
        if {*bond} == {route[i], route[i+1]}:
            break
    print(route)
    
    if i == 0:
        short_site = [route[0], route[i+2]]
    elif i + 1 == len(route) - 1:
        short_site = [route[i-1], route[i+1]]
    else:
        short_site = [route[i-1], route[i+2]]
    
    bond_pos1 = atoms.get_positions()[bond[0]]
    bond_pos2 = atoms.get_positions()[bond[1]]
    bond_vector = [float(bond_pos2[0]-bond_pos1[0]),\
                   float(bond_pos2[1]-bond_pos1[1]),\
                   float(bond_pos2[2]-bond_pos1[2])]
    print(bond_vector)
    
    short_pos1 = atoms.get_positions()[short_site[0]]
    short_pos2 = atoms.get_positions()[short_site[1]]
    short_angle_vector = [float(short_pos2[0]-short_pos1[0]),\
                          float(short_pos2[1]-short_pos1[1]),\
                          float(short_pos2[2]-short_pos1[2])]
    
    #print(bond_vector, short_angle_vector)
    cos = cos_angle(bond_vector, short_angle_vector)
    
    return cos

In [15]:
import pandas as pd

#csv = pd.read_csv('Force_candidate.csv')
csv = pd.read_csv('Force_candidate_transfer.csv')
#csv = pd.read_csv('candidate_bonds_no_conformer.csv')

Molnums0 = csv['Molecule'].tolist()

cand_bonds1 = csv['Atom 1'].tolist()
cand_bonds2 = csv['Atom 2'].tolist()

w = open('Force_candidate_angle.csv','w')
w.write("Molecule,Atom 1,Atom 2,Cos Part Angles,Cos(Length)\n")

j = 0
for turn,num in enumerate(Molnums0):
    print('Molecule {0}:'.format(num))
    atoms = read('./Extra/{0}_0.0.xyz'.format(num))
    bond = [int(cand_bonds1[turn]), int(cand_bonds2[turn])]
    length = atoms.get_distance(int(cand_bonds1[turn]), int(cand_bonds2[turn]))
    
    if type(mols[j]) != str:
        mol0 = str(mols[j])
    else:
        mol0 = mols[j]
    
    if type(num) != str:
        num = str(num)
    
    while mol0 != num:
        #print(mol0, num)
        j = j + 1
        mol0 = mols[j]
    
    route = lists[j]
    cos = part_angle_calc(bond, route, atoms)
    print('Bond {0}-{1} cos(theta): {2}'.format(*bond, cos))
    print()
    w.write('{0},{1},{2},{3},{4}\n'.format(num, *bond,  cos, cos*length))
    
w.close()

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[-0.9411624510803005, 1.1172205705080103, -0.5274476674749802]
Bond 4-5 cos(theta): 0.684360938109942

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[0.22447868544323013, 0.23521973516018946, -1.50780503066057]
Bond 3-4 cos(theta): 0.809900014135254

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[1.0798325709392596, 0.9933228326780705, -0.3085430053019005]
Bond 0-3 cos(theta): 0.7302368461145762

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[0.6562039834232603, 0.7182525583534902, -1.1823078142604297]
Bond 5-16 cos(theta): 0.685837059117859

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[0.18698840132367067, 0.49627730303175976, -1.3991638269679996]
Bond 7-0 cos(theta): 0.7385327456020444

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[0.5027290671616198, 0.7463699476880099, -1.1165185066905798]
Bond 18-19 cos(theta): 0.860435080751254

Molecule 4001:
[9, 8, 7, 0, 3, 4, 5, 16, 18, 19]
[0.26017131338179933, 0.5643917671907

In [65]:
#csv = pd.read_csv('all.csv')
csv = pd.read_csv('2d_transfer.csv')

Molnums0 = csv['Molecule'].tolist()

Broken_bonds1_1 = csv['atom1'].tolist()
Broken_bonds2_1 = csv['atom2'].tolist()
Broken_bonds1_2 = csv['atom1_2'].tolist()
Broken_bonds2_2 = csv['atom2_2'].tolist()
Broken_bonds1_3 = csv['atom1_3'].tolist()
Broken_bonds2_3 = csv['atom2_3'].tolist()
Broken_bonds1_4 = csv['atom1_4'].tolist()
Broken_bonds2_4 = csv['atom2_4'].tolist()

w = open('Force_candidate_angle_mol.csv','w')
w.write("Molecule,Cos_bond1,Cos_bond2,Cos_bond3,Cos_bond4\n")

loc = './Extra/'

j = 0
for turn,num in enumerate(Molnums0):
    
    print('Molecule {0}:'.format(num))
    while num != mols[j]:
        print('Comparation:', num, mols[j])
        j = j + 1
    
    bondbox = allbrokenbonds(num)
    atoms = read(loc+f'{num}_0.0.xyz')
    w.write('{0},'.format(num))
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    
    route = lists[j]
    print('Reaction Bond 1:', bondnum)
    cos = part_angle_calc(bondnum, route, atoms)
    length = atoms.get_distance(*bondnum)
    print('cos(theta):', cos)
    w.write('{0},{1},'.format(cos,cos*length))
    try:
        bondnum = bondbox[1]
        print('Reaction Atoms 2:', bondnum)
        cos = part_angle_calc(bondnum, route, atoms)
        length = atoms.get_distance(*bondnum)
        print('cos(theta):', cos)
        w.write('{0},{1},'.format(cos,cos*length))
        try:
            bondnum = bondbox[2]
            print('Reaction Atoms 3:', bondnum)
            cos = part_angle_calc(bondnum, route, atoms)
            length = atoms.get_distance(*bondnum)
            print('cos(theta):', cos)
            w.write('{0},{1},'.format(cos,cos*length))
            try:
                bondnum = bondbox[3]
                print('Reaction Atoms 4:', bondnum)
                cos = part_angle_calc(bondnum, route, atoms)
                length = atoms.get_distance(*bondnum)
                print('cos(theta):', cos)
                w.write('{0},{1},'.format(cos,cos*length))
            except:
                pass
        except:
            pass
    except:
        pass
    w.write('\n')
    print()

w.close()

Molecule 4001:
Molecule 4003:
Comparation: 4003 4001
Molecule 4005:
Comparation: 4005 4003
Reaction Bond 1: [6, 2]
[27, 12, 7, 6, 2, 16, 41]
[0.12017715592976064, -1.4981068222793406, 0.52485181696482]
cos(theta): 0.9944721338506327

Molecule 4006:
Comparation: 4006 4005
Molecule 4007:
Comparation: 4007 4006
Molecule 4008:
Comparation: 4008 4007
Reaction Bond 1: [2, 0]
[14, 12, 2, 0, 18, 19, 20]
[1.07434811087926, -0.10250360757449961, -1.09792687003194]
cos(theta): 0.9338717559679588

Molecule 4009:
Comparation: 4009 4008
Reaction Bond 1: [7, 8]
[20, 17, 7, 8, 16, 24]
[0.3676061737960099, -0.15571226016132, -1.46273159622412]
cos(theta): 0.9995630392467769

Molecule 4009-2:
Comparation: 4009-2 4009
Reaction Bond 1: [7, 8]
[20, 17, 7, 8, 16, 24]
[-0.99052664735511, -0.63511691941962, -0.9559455697234602]
cos(theta): 0.9996843373384283

Molecule 4010:
Comparation: 4010 4009-2
Reaction Bond 1: [7, 8]
[25, 16, 13, 7, 8, 12, 17, 29]
[0.27942564155672, -0.038853711663900004, -1.503972485800

### Bond Type

In [17]:
def bond_type(eleneg1, eleneg2):
    if {eleneg1, eleneg2} == {2.55, 2.55}:
        return 0 # C-C
    elif {eleneg1, eleneg2} == {2.55, 3.44}:
        return 1 # C-O
    elif {eleneg1, eleneg2} == {2.55, 3.04}:
        return 2 # C-N
    elif {eleneg1, eleneg2} == {3.44, 3.44}:
        return 3 # O-O
    elif {eleneg1, eleneg2} == {3.04, 3.04}:
        return 4 # N-N
    elif {eleneg1, eleneg2} == {3.44, 3.04}:
        return 5 # N-O
    elif {eleneg1, eleneg2} == {2.58, 2.58}:
        return 6 # S-S
    elif {eleneg1, eleneg2} == {2.58, 2.55}:
        return 7 # S-C
    elif {eleneg1, eleneg2} == {2.58, 3.44}:
        return 8 # S-O
    elif {eleneg1, eleneg2} == {2.58, 3.04}:
        return 9 # S-N
    else:
        raise Exception('No such bond type!')

#csv = pd.read_csv('candidate_bonds_no_conformer.csv')
csv = pd.read_csv('Force_candidate_transfer3.csv')
#csv = pd.read_csv('all_transfer.csv')
#csv = pd.read_csv('./generalization/final_isbroken_test_transfer.csv')

Molnums= csv['Molecule'].tolist()

#cand_bonds1 = csv['Atom 1'].tolist()
#cand_bonds2 = csv['Atom 2'].tolist()
eleneg1 = csv['Elecneg_1'].tolist()
eleneg2 = csv['Elecneg_2'].tolist()

w = open('bondtype.csv','w')
w.write("Molecule,Bondtype\n")

loc = './Extra/ir/xyz/'

j = 0
for turn,num in enumerate(Molnums):
    try:
        bt = bond_type(eleneg1[turn], eleneg2[turn])
    except:
        w.write('{0},ERROR\n'.format(num))
        continue
    
    #print('Atom {0}: {1}'.format(atom1, mull_charge[atom1]))
    #print('Atom {0}: {1}'.format(atom2, mull_charge[atom2]))
    #print()
    
    w.write('{0},{1}\n'.format(num, bt))
    
w.close()

### Mulliken Charge

In [14]:
#csv = pd.read_csv('Force_candidate_transfer3.csv')
csv = pd.read_csv('./generalization/final_isbroken_test_orig.csv')


Molnums0 = csv['Molecule'].tolist()

cand_bonds1 = csv['Atom 1'].tolist()
cand_bonds2 = csv['Atom 2'].tolist()

w = open('Force_candidate_charge.csv','w')
w.write("Molecule,charge_atom1,charge_atom2\n")

loc = './Extra/ir/out/'

j = 0
for turn,num in enumerate(Molnums0):
    mull_charge = []
    print('Molecule {0}:'.format(num))
    
    if type(num) != str:
        num = str(num)
    
    f1 = open(loc+f'{num}_0.0.out','r')
    signal = 0
    for line in f1:
        if signal == 2:
            a3 = re.search('Sum of atomic charges', line)
            if a3:
                signal = 0
                continue
            a2 = re.search(r'\-?[0-9]+\.[0-9]+', line)
            if a2:
                mull_charge.append(eval(a2.group(0)))
        if signal == 1:
            mull_charge = []
            a1 = re.search('-----------------------', line)
            if a1:
                signal = 2
        if signal == 0:
            a0 = re.search('MULLIKEN ATOMIC CHARGES', line)
            if a0:
                signal = 1
    f1.close()
    w.write('{0},'.format(num))
    print(len(mull_charge))
    
    atom1 = int(cand_bonds1[turn])
    atom2 = int(cand_bonds2[turn])

    print('Atom {0}: {1}'.format(atom1, mull_charge[atom1]))
    print('Atom {0}: {1}'.format(atom2, mull_charge[atom2]))
    print()
    w.write('{0},{1},{2}\n'.format(num, mull_charge[atom1], mull_charge[atom2]))
    
w.close()

Molecule 1:
42
Atom 2: -0.064145
Atom 4: -0.077219

Molecule 1:
42
Atom 20: 0.222945
Atom 21: 0.192931

Molecule 1:
42
Atom 17: 0.211758
Atom 16: 0.213321

Molecule 1:
42
Atom 4: -0.077219
Atom 11: 0.202643

Molecule 1:
42
Atom 0: 0.211492
Atom 2: -0.064145

Molecule 1:
42
Atom 35: 0.119778
Atom 36: 0.050022

Molecule 1:
42
Atom 31: 0.050978
Atom 30: 0.118058

Molecule 1:
42
Atom 15: -0.297731
Atom 20: 0.222945

Molecule 1:
42
Atom 16: 0.213321
Atom 1: -0.311833

Molecule 1:
42
Atom 21: 0.192931
Atom 27: -0.328512

Molecule 1:
42
Atom 24: -0.327807
Atom 17: 0.211758

Molecule 1:
42
Atom 30: 0.118058
Atom 24: -0.327807

Molecule 1:
42
Atom 27: -0.328512
Atom 35: 0.119778

Molecule 1:
42
Atom 11: 0.202643
Atom 15: -0.297731

Molecule 1:
42
Atom 1: -0.311833
Atom 0: 0.211492

Molecule 2:
42
Atom 2: -0.075748
Atom 4: -0.07726

Molecule 2:
42
Atom 17: 0.210292
Atom 16: 0.214366

Molecule 2:
42
Atom 20: 0.214784
Atom 21: 0.210321

Molecule 2:
42
Atom 4: -0.07726
Atom 11: 0.206172

Molecule 2

In [12]:
5

5

In [52]:
#csv = pd.read_csv('all_advanced.csv')
csv = pd.read_csv('2d_transfer.csv')
#csv = pd.read_csv('./generalization/final_isbroken_test_transfer.csv')

Molnums0 = csv['Molecule'].tolist()

w = open('charge.csv','w')
w.write("Molecule,charge_atom1_1,charge_atom2_1,charge_atom1_2,charge_atom2_2,charge_atom1_3,charge_atom2_3,charge_atom1_4,charge_atom2_4\n")

loc = './Extra/ir/out/'

j = 0

for turn,num in enumerate(Molnums0):
    mull_charge = []
    print('Molecule {0}:'.format(num))
    #while num != mols[j]:
    #    print('Comparation:', num, mols[j])
    #    j = j + 1
    f1 = open(loc+f'{num}_0.0.out','r')
    
    signal = 0
    for line in f1:
        if signal == 2:
            a3 = re.search('Sum of atomic charges', line)
            if a3:
                signal = 0
                continue
            a2 = re.search(r'\-?[0-9]+\.[0-9]+', line)
            if a2:
                mull_charge.append(eval(a2.group(0)))
        if signal == 1:
            a1 = re.search('-----------------------', line)
            if a1:
                signal = 2
        if signal == 0:
            a0 = re.search('MULLIKEN ATOMIC CHARGES', line)
            if a0:
                signal = 1
    f1.close()
    
    w.write('{0},'.format(num))
    
    bondbox = allbrokenbonds(num)
    #atoms = read(loc+f'{num}_0.0.xyz')
    
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    atom1 = bondnum[0]
    atom2 = bondnum[1]
    
    print('Reaction Bond 1:', mull_charge[atom1], mull_charge[atom2])
    w.write('{0:.4f},{1:.4f},'.format(mull_charge[atom1], mull_charge[atom2]))
    try:
        bondnum = bondbox[1]
        atom1 = bondnum[0]
        atom2 = bondnum[1]
        print('Reaction Bond 2:', mull_charge[atom1], mull_charge[atom2])
        w.write('{0:.4f},{1:.4f},'.format(mull_charge[atom1], mull_charge[atom2]))
        try:
            bondnum = bondbox[2]
            atom1 = bondnum[0]
            atom2 = bondnum[1]
            print('Reaction Bond 3:', mull_charge[atom1], mull_charge[atom2])
            w.write('{0:.4f},{1:.4f},'.format(mull_charge[atom1], mull_charge[atom2]))
            try:
                bondnum = bondbox[3]
                atom1 = bondnum[0]
                atom2 = bondnum[1]
                print('Reaction Bond 4:', mull_charge[atom1], mull_charge[atom2])
                w.write('{0:.4f},{1:.4f},'.format(mull_charge[atom1], mull_charge[atom2]))
            except:
                pass
        except:
            pass
    except:
        pass
    w.write('\n')
    print()

w.close()

Molecule 4001:
Molecule 4003:
Molecule 4005:
Reaction Bond 1: 0.073187 0.08321

Molecule 4006:
Molecule 4007:
Molecule 4008:
Reaction Bond 1: 0.035224 -0.046271

Molecule 4009:
Reaction Bond 1: -0.212863 -0.144286

Molecule 4009-2:
Reaction Bond 1: -0.216384 -0.14692

Molecule 4010:
Reaction Bond 1: -0.065954 -0.177519

Molecule 4010-2:
Reaction Bond 1: -0.129427 -0.153306

Molecule 4011:
Molecule 4012:
Reaction Bond 1: -0.125773 -0.107769

Molecule 4012-2:
Reaction Bond 1: -0.124883 -0.131056

Molecule 4013:
Reaction Bond 1: 0.002442 -0.096516

Molecule 4014:
Reaction Bond 1: 0.098085 -0.054588

Molecule 4015:
Reaction Bond 1: -0.056341 -0.094229

Molecule 4016:
Reaction Bond 1: -0.305404 0.229245

Molecule 4017:
Reaction Bond 1: -0.313004 0.23876

Molecule 4018:
Reaction Bond 1: -0.320219 0.213156

Molecule 4019:
Reaction Bond 1: 0.246154 -0.29682

Molecule 4020:
Reaction Bond 1: 0.233128 -0.300694

Molecule 4021:
Reaction Bond 1: -0.318304 0.190744

Molecule 4022:
Molecule 4023:
Rea

In [53]:
#csv = pd.read_csv('all_advanced.csv')
csv = pd.read_csv('all_transfer.csv')

Molnums0 = csv['Molecule'].tolist()

w = open('charge.csv','w')
w.write("Molecule,charge_atom1_1,charge_atom2_1,charge_atom1_2,charge_atom2_2,charge_atom1_3,charge_atom2_3,charge_atom1_4,charge_atom2_4\n")

loc = './Extra/ir/out/'

j = 0

for turn,num in enumerate(Molnums0):
    mull_charge = []
    print('Molecule {0}:'.format(num))
    #while num != mols[j]:
    #    print('Comparation:', num, mols[j])
    #    j = j + 1
    f1 = open(loc+f'{num}_0.0.out','r')
    
    signal = 0
    for line in f1:
        if signal == 2:
            a3 = re.search('Sum of atomic charges', line)
            if a3:
                signal = 0
                continue
            a2 = re.search(r'\-?[0-9]+\.[0-9]+', line)
            if a2:
                mull_charge.append(eval(a2.group(0)))
        if signal == 1:
            a1 = re.search('-----------------------', line)
            if a1:
                signal = 2
        if signal == 0:
            a0 = re.search('MULLIKEN ATOMIC CHARGES', line)
            if a0:
                signal = 1
    f1.close()
    
    w.write('{0},'.format(num))
    
    bondbox = allbrokenbonds(num)
    #atoms = read(loc+f'{num}_0.0.xyz')
    
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    atom1 = bondnum[0]
    atom2 = bondnum[1]
    charge_box = [mull_charge[atom1], mull_charge[atom2]]
    print('Reaction Bond 1:', min(charge_box), max(charge_box))
    w.write('{0:.4f},{1:.4f},'.format(min(charge_box), max(charge_box)))
    try:
        bondnum = bondbox[1]
        atom1 = bondnum[0]
        atom2 = bondnum[1]
        charge_box = [mull_charge[atom1], mull_charge[atom2]]
        print('Reaction Bond 2:', min(charge_box), max(charge_box))
        w.write('{0:.4f},{1:.4f},'.format(min(charge_box), max(charge_box)))
        try:
            bondnum = bondbox[2]
            atom1 = bondnum[0]
            atom2 = bondnum[1]
            charge_box = [mull_charge[atom1], mull_charge[atom2]]
            print('Reaction Bond 3:', min(charge_box), max(charge_box))
            w.write('{0:.4f},{1:.4f},'.format(min(charge_box), max(charge_box)))
            try:
                bondnum = bondbox[3]
                atom1 = bondnum[0]
                atom2 = bondnum[1]
                charge_box = [mull_charge[atom1], mull_charge[atom2]]
                print('Reaction Bond 4:', min(charge_box), max(charge_box))
                w.write('{0:.4f},{1:.4f},'.format(min(charge_box), max(charge_box)))
            except:
                pass
        except:
            pass
    except:
        pass
    w.write('\n')
    print()

w.close()

Molecule 4001:
Molecule 4003:
Molecule 4005:
Reaction Bond 1: 0.073187 0.08321

Molecule 4006:
Molecule 4008:
Reaction Bond 1: -0.046271 0.035224

Molecule 4009:
Reaction Bond 1: -0.212863 -0.144286

Molecule 4009-2:
Reaction Bond 1: -0.216384 -0.14692

Molecule 4010:
Reaction Bond 1: -0.177519 -0.065954

Molecule 4010-2:
Reaction Bond 1: -0.153306 -0.129427

Molecule 4011:
Molecule 4012:
Reaction Bond 1: -0.125773 -0.107769

Molecule 4012-2:
Reaction Bond 1: -0.131056 -0.124883

Molecule 4013:
Reaction Bond 1: -0.096516 0.002442

Molecule 4014:
Reaction Bond 1: -0.054588 0.098085

Molecule 4015:
Reaction Bond 1: -0.094229 -0.056341

Molecule 4016:
Reaction Bond 1: -0.305404 0.229245

Molecule 4017:
Reaction Bond 1: -0.313004 0.23876

Molecule 4018:
Reaction Bond 1: -0.320219 0.213156

Molecule 4019:
Reaction Bond 1: -0.29682 0.246154

Molecule 4020:
Reaction Bond 1: -0.300694 0.233128

Molecule 4021:
Reaction Bond 1: -0.318304 0.190744

Molecule 4022:
Molecule 4023:
Reaction Bond 1: -

In [55]:
# with min and max

#csv = pd.read_csv('all_advanced.csv')
csv = pd.read_csv('2d_transfer.csv')

Molnums0 = csv['Molecule'].tolist()

Broken_bonds1_1 = csv['atom1'].tolist()
Broken_bonds2_1 = csv['atom2'].tolist()
Broken_bonds1_2 = csv['atom1_2'].tolist()
Broken_bonds2_2 = csv['atom2_2'].tolist()
Broken_bonds1_3 = csv['atom1_3'].tolist()
Broken_bonds2_3 = csv['atom2_3'].tolist()
Broken_bonds1_4 = csv['atom1_4'].tolist()
Broken_bonds2_4 = csv['atom2_4'].tolist()

w = open('charge_minmax.csv','w')
w.write("Molecule,charge_atom_max_1,charge_atom_min_1,charge_atom_max_2,charge_atom_min_2,charge_atom_max_3,charge_atom_min_3,charge_atom_max_4,charge_atom_min_4\n")

loc = './Extra/ir/out/'

j = 0

for turn,num in enumerate(Molnums0):
    mull_charge = []
    print('Molecule {0}:'.format(num))
    while num != mols[j]:
        print('Comparation:', num, mols[j])
        j = j + 1
    f1 = open(loc+f'{num}_0.0.out','r')
    
    signal = 0
    for line in f1:
        if signal == 2:
            a3 = re.search('Sum of atomic charges', line)
            if a3:
                signal = 0
                continue
            a2 = re.search(r'\-?[0-9]+\.[0-9]+', line)
            if a2:
                mull_charge.append(eval(a2.group(0)))
        if signal == 1:
            a1 = re.search('-----------------------', line)
            if a1:
                signal = 2
        if signal == 0:
            a0 = re.search('MULLIKEN ATOMIC CHARGES', line)
            if a0:
                signal = 1
    f1.close()
    
    w.write('{0},'.format(num))
    
    bondbox = allbrokenbonds(num)
    #atoms = read(loc+f'{num}_0.0.xyz')
    
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    atom1 = bondnum[0]
    atom2 = bondnum[1]
    print('Reaction Bond 1:', mull_charge[atom1], mull_charge[atom2])
    mull_charge_box = [mull_charge[atom1], mull_charge[atom2]]
    w.write('{0:.4f},{1:.4f},'.format(min(mull_charge_box), max(mull_charge_box)))
    try:
        bondnum = bondbox[1]
        atom1 = bondnum[0]
        atom2 = bondnum[1]
        print('Reaction Bond 2:', mull_charge[atom1], mull_charge[atom2])
        mull_charge_box = [mull_charge[atom1], mull_charge[atom2]]
        w.write('{0:.4f},{1:.4f},'.format(min(mull_charge_box), max(mull_charge_box)))
        try:
            bondnum = bondbox[2]
            atom1 = bondnum[0]
            atom2 = bondnum[1]
            print('Reaction Bond 3:', mull_charge[atom1], mull_charge[atom2])
            mull_charge_box = [mull_charge[atom1], mull_charge[atom2]]
            w.write('{0:.4f},{1:.4f},'.format(min(mull_charge_box), max(mull_charge_box)))
            try:
                bondnum = bondbox[3]
                atom1 = bondnum[0]
                atom2 = bondnum[1]
                print('Reaction Bond 4:', mull_charge[atom1], mull_charge[atom2])
                mull_charge_box = [mull_charge[atom1], mull_charge[atom2]]
                w.write('{0:.4f},{1:.4f},'.format(min(mull_charge_box), max(mull_charge_box)))
            except:
                pass
        except:
            pass
    except:
        pass
    w.write('\n')
    print()

w.close()

Molecule 4001:
Molecule 4003:
Comparation: 4003 4001
Molecule 4005:
Comparation: 4005 4003
Reaction Bond 1: 0.073187 0.08321

Molecule 4006:
Comparation: 4006 4005
Molecule 4007:
Comparation: 4007 4006
Molecule 4008:
Comparation: 4008 4007
Reaction Bond 1: 0.035224 -0.046271

Molecule 4009:
Comparation: 4009 4008
Reaction Bond 1: -0.212863 -0.144286

Molecule 4009-2:
Comparation: 4009-2 4009
Reaction Bond 1: -0.216384 -0.14692

Molecule 4010:
Comparation: 4010 4009-2
Reaction Bond 1: -0.065954 -0.177519

Molecule 4010-2:
Comparation: 4010-2 4010
Reaction Bond 1: -0.129427 -0.153306

Molecule 4011:
Comparation: 4011 4010-2
Molecule 4012:
Comparation: 4012 4011
Reaction Bond 1: -0.125773 -0.107769

Molecule 4012-2:
Comparation: 4012-2 4012
Reaction Bond 1: -0.124883 -0.131056

Molecule 4013:
Comparation: 4013 4012-2
Reaction Bond 1: 0.002442 -0.096516

Molecule 4014:
Comparation: 4014 4013
Reaction Bond 1: 0.098085 -0.054588

Molecule 4015:
Comparation: 4015 4014
Reaction Bond 1: -0.0563

### Energy

In [90]:
csv = pd.read_csv('all_advanced.csv')
Molnums = csv['Molecule'].tolist()

w = open('E.csv','w')
w.write("Molecule,E_0.0\n")

loc = './Extra/ir/out/'

j = 0
for turn,num in enumerate(Molnums):
    mull_charge = []
    print('Molecule {0}:'.format(num))
    
    if type(num) != str:
        num = str(num)
    
    f1 = open(loc+f'{num}_0.0.out','r')

    for line in f1:
        a0 = re.search('FINAL SINGLE POINT ENERGY', line)
        if a0:
            E = re.search(r'\-[0-9]+\.[0-9]+', line)
            if E:
                e = E.group(0)
    f1.close()

    print('Potential Energy: {0}'.format(e))
    print()
    w.write('{0},{1}\n'.format(num, e))
    
w.close()

Molecule 1:
Potential Energy: -1145.882181870673

Molecule 2:
Potential Energy: -1145.882155122968

Molecule 3:
Potential Energy: -1237.997949362380

Molecule 4:
Potential Energy: -1237.996547523193

Molecule 5:
Potential Energy: -1330.110179874944

Molecule 6:
Potential Energy: -1330.106740937464

Molecule 7:
Potential Energy: -1185.645141947633

Molecule 8:
Potential Energy: -1528.853679230150

Molecule 9:
Potential Energy: -1528.844893799561

Molecule 10:
Potential Energy: -1262.725359381919

Molecule 11:
Potential Energy: -1881.837097365115

Molecule 14:
Potential Energy: -1221.511821455915

Molecule 15:
Potential Energy: -1833.730647290422

Molecule 16:
Potential Energy: -773.063580659442

Molecule 17:
Potential Energy: -906.837344912690

Molecule 18:
Potential Energy: -1092.840704075743

Molecule 19:
Potential Energy: -1001.813281534411

Molecule 20:
Potential Energy: -1416.618497519960

Molecule 21:
Potential Energy: -1301.716413375762

Molecule 22:
Potential Energy: -1301.71392

## Translation between Molecule csv and Bond csv

In [23]:
csv = pd.read_csv('Force_candidate_transfer.csv')

Molnums = csv['Molecule'].tolist()

IsBroken = csv['IsBroken'].tolist()

BLs = csv['Bond Length'].tolist()
SBLs = csv['Standardised Bond Length'].tolist()
cand_bonds1 = csv['Atom 1'].tolist()
cand_bonds2 = csv['Atom 2'].tolist()
BondInRings = csv['InRing_B'].tolist()

EC1 = csv['Elecneg_1'].tolist()
EC2 = csv['Elecneg_2'].tolist()

charge1 = csv['Charge_1'].tolist()
charge2 = csv['Charge_2'].tolist()

InConj1 = csv['InConj_1'].tolist()
InConj2 = csv['InConj_2'].tolist()

InRing1 = csv['InConj_1'].tolist()
InRing2 = csv['InConj_2'].tolist()

KeyError: 'Charge_1'

In [68]:
import pandas as pd
import os
import re

def allbrokenbonds(mol, file='Force_candidate_transfer.csv',output='Atom'):
    csv = pd.read_csv(file)

    Molnums = csv['Molecule'].tolist()

    IsBroken = csv['IsBroken'].tolist()
    try:
        MayerBond = csv['Mayer'].tolist()
    except:
        pass
    try:
        Lengths = csv['Bond Length'].tolist()
        ST_Lengths = csv['Standardised Bond Length'].tolist()
    except:
        pass

    cand_bonds1 = csv['Atom 1'].tolist()
    cand_bonds2 = csv['Atom 2'].tolist()

    #Elecneg1 = csv['Elecneg_1'].tolist()
    #Elecneg2 = csv['Elecneg_2'].tolist()
    #Elec_reac1 = csv['Elec_reac1'].tolist()
    #Elec_reac2 = csv['Elec_reac2'].tolist()
    #Nuc_reac1 = csv['Elec_reac1'].tolist()
    #Nuc_reac2 = csv['Elec_reac2'].tolist()
    #Mull_charge1 = csv['charge_atom1'].tolist()
    #Mull_charge2 = csv['charge_atom2'].tolist()
    cos_angle = csv['Cos Part Angles'].tolist()
    cos_length = csv['Cos(Length)'].tolist()

    outputs = []
    for i, mol0 in enumerate(Molnums):
        if mol0 == mol:
            if IsBroken[i] == 1:
                if output == 'Atom':
                    outputs.append([cand_bonds1[i], cand_bonds2[i]])
                elif output == 'Mayer':
                    outputs.append([cand_bonds1[i], cand_bonds2[i], MayerBond[i]])
                elif output == 'Length':
                    outputs.append([cand_bonds1[i], cand_bonds2[i], Lengths[i], ST_Lengths[i]])
                elif output == 'Charge':
                    Elecneg = [Elecneg1[i], Elecneg2[i]]
                    Elecnegmaxmin = [max(Elecneg), min(Elecneg)]#, max(Elecneg) - min(Elecneg)]
                    #Elec_reac = [Elec_reac1[i], Elec_reac2[i]]
                    #Elec_reacmaxmin = [max(Elec_reac), min(Elec_reac)]#, max(Elec_reac) - min(Elec_reac)]
                    #Nuc_reac = [Nuc_reac1[i], Nuc_reac2[i]]
                    #Nuc_reacmaxmin = [max(Nuc_reac), min(Nuc_reac)]#, max(Nuc_reac) - min(Nuc_reac)]
                    Mull_charge = [Mull_charge1[i], Mull_charge2[i]]
                    Mull_chargemaxmin = [max(Mull_charge), min(Mull_charge)]#, max(Mull_charge) - min(Mull_charge)]
                    #outputs.append([*Elecnegmaxmin, *Elec_reacmaxmin, *Nuc_reacmaxmin, *Mull_chargemaxmin])
                    outputs.append([*Elecnegmaxmin, *Mull_chargemaxmin])
                elif output == 'cos':
                    outputs.append([cand_bonds1[i], cand_bonds2[i], cos_angle[i], cos_length[i]])
    return outputs

In [69]:
csv = pd.read_csv('all_transfer.csv')
Molnums = csv['Molecule'].tolist()

loc = './Extra/'

w = open('standardised_bonds_transfer.csv','w')
w.write("Molecule,Ele\n")


for turn,num in enumerate(Molnums):
    bondbox = allbrokenbonds(num, output='Length')
    atoms = read(loc+f'{num}_0.0.xyz')
    w.write('{0},'.format(num))
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
    #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
    #bondele = [ele1, ele2]
    print('Molecule {0}:'.format(num))
    print('Reaction Atoms 1:', bondnum)
    #print('Mayer:', bondnum[-1])
    w.write('{0},{1},'.format(bondnum[-2], bondnum[-1]))
    try:
        bondnum = bondbox[1]
        #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
        #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
        #bondele = [ele1, ele2]
        print('Reaction Atoms 2:', bondnum)
        #print('Mayer:', bondnum[-1])
        w.write('{0},{1},'.format(bondnum[-2], bondnum[-1]))
        try:
            bondnum = bondbox[2]
            #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
            #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
            #bondele = [ele1, ele2]
            print('Reaction Atoms 3:', bondnum)
            #print('Mayer:', bondnum[-1])
            w.write('{0},{1},'.format(bondnum[-2], bondnum[-1]))
            try:
                bondnum = bondbox[3]
                #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
                #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
                #bondele = [ele1, ele2]
                print('Reaction Atoms 4:', bondnum)
                #print('Mayer:', bondnum[-1])
                w.write('{0},{1},'.format(bondnum[-2], bondnum[-1]))
            except:
                pass
        except:
            pass
    except:
        pass
    print()
    w.write('\n')

w.close()

Molecule 4005:
Reaction Atoms 1: [6, 2, 1.5919284, 1.041837958]

Molecule 4008:
Reaction Atoms 1: [2, 0, 1.539537029, 1.007550412]

Molecule 4009:
Reaction Atoms 1: [7, 8, 1.5162336, 0.992298946]

Molecule 4009-2:
Reaction Atoms 1: [7, 8, 1.5160305, 0.992167256]

Molecule 4010:
Reaction Atoms 1: [7, 8, 1.530203104, 1.001441822]

Molecule 4010-2:
Reaction Atoms 1: [7, 8, 1.536073556, 1.005283741]

Molecule 4012:
Reaction Atoms 1: [7, 8, 1.530231295, 1.001460271]

Molecule 4012-2:
Reaction Atoms 1: [7, 8, 1.511521716, 0.989215783]

Molecule 4013:
Reaction Atoms 1: [4, 1, 1.529294788, 1.000847374]

Molecule 4014:
Reaction Atoms 1: [1, 4, 1.504434584, 0.984577608]

Molecule 4015:
Reaction Atoms 1: [4, 1, 1.514426708, 0.991116956]

Molecule 4016:
Reaction Atoms 1: [18, 17, 1.467554783, 1.043038225]

Molecule 4017:
Reaction Atoms 1: [18, 17, 1.463539808, 1.040184654]

Molecule 4018:
Reaction Atoms 1: [20, 19, 1.467275349, 1.042839623]

Molecule 4019:
Reaction Atoms 1: [17, 18, 1.431997479, 1

In [54]:
csv = pd.read_csv('all_transfer.csv')
Molnums = csv['Molecule'].tolist()

loc = './Extra/'

w = open('Charge.csv','w')
w.write("Molecule,Ele\n")


for turn,num in enumerate(Molnums):
    bondbox = allbrokenbonds(num, output='Charge')
    atoms = read(loc+f'{num}_0.0.xyz')
    w.write('{0},'.format(num))
    if len(bondbox) == 0:
        w.write('\n')
        continue
    bondnum = bondbox[0]
    #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
    #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
    #bondele = [ele1, ele2]
    print('Molecule {0}:'.format(num))
    #print('Reaction Atoms 1:', bondnum)
    #print('Mayer:', bondnum[-1])
    w.write('{0},{1},'.format(bondnum[-4], bondnum[-3]))
    try:
        bondnum = bondbox[1]
        #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
        #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
        #bondele = [ele1, ele2]
        #print('Reaction Atoms 2:', bondnum)
        #print('Mayer:', bondnum[-1])
        w.write('{0},{1},'.format(bondnum[-4], bondnum[-3]))
        try:
            bondnum = bondbox[2]
            #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
            #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
            #bondele = [ele1, ele2]
            #print('Reaction Atoms 3:', bondnum)
            #print('Mayer:', bondnum[-1])
            w.write('{0},{1},'.format(bondnum[-4], bondnum[-3]))
            try:
                bondnum = bondbox[3]
                #ele1 = atoms.get_chemical_symbols()[bondnum[0]]
                #ele2 = atoms.get_chemical_symbols()[bondnum[1]]
                #bondele = [ele1, ele2]
                #print('Reaction Atoms 4:', bondnum)
                #print('Mayer:', bondnum[-1])
                w.write('{0},{1},'.format(bondnum[-4], bondnum[-3]))
            except:
                pass
        except:
            pass
    except:
        pass
    print()
    w.write('\n')

w.close()

Molecule 4005:

Molecule 4008:

Molecule 4009:

Molecule 4009-2:

Molecule 4010:

Molecule 4010-2:

Molecule 4012:

Molecule 4012-2:

Molecule 4013:

Molecule 4014:

Molecule 4015:

Molecule 4016:

Molecule 4017:

Molecule 4018:

Molecule 4019:

Molecule 4020:

Molecule 4021:

Molecule 4023:

Molecule 4024-:

Molecule 4025:

Molecule 4026:

Molecule 4027:

Molecule 4028:

Molecule 4029:

Molecule 4030:

Molecule 4032:

Molecule 4033:

Molecule 4034:

Molecule 4035:

Molecule 4037:

Molecule 4038:

Molecule 4039:

Molecule 4040:

Molecule 4041:

Molecule 4042:

Molecule 4043:

Molecule 4044:

Molecule 4045:

Molecule 4046:

Molecule 4047:

Molecule 4048:

Molecule 4049:

Molecule 4050:

Molecule 4051:

Molecule 4052:

Molecule 4053:

Molecule 4054:

Molecule 4055:

Molecule 4057:

Molecule 4058:

Molecule 4059:

Molecule 4060:

Molecule 4061:

Molecule 4061-:

Molecule 4062:

Molecule 4063:

Molecule 4064:

Molecule 4065:

Molecule 4066:

Molecule 4067:

Molecule 4068:

Molecule 4068-2: